### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from tabulate import tabulate
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack
import astropy.units as u
import numpy as np

### Options

In [ ]:
export_all = False
fits_path = '../output/asterisms-2025-07'
fov = 2*u.arcmin

### Load Asterisms

In [ ]:
fits_file = f"{fits_path}/asterisms-GNAO.fits"
asterisms = Table.read(fits_file, format='fits')
print('Number of asterisms:', len(asterisms))

### Load Targets

In [ ]:
# Load targets
target_mode = 3.5
match target_mode:
    case 1: # 3D-HST Sample Targets
        targets_name = 'sample-targets'
        fits_file = f"{fits_path}/sample-targets.fits"
        targets = Table.read(fits_file, format='fits')
        col_formats = ["", ".0f", ".5f", ".5f", ".3f", ".1f", ".1f", ".1f"]
    case 2: # Clusters for Lamiya
        targets = Table.read("../data/girmos-sci/lamiya_clusters.csv", format='csv')
        targets_name = 'lamiya_clusters'
        targets.rename_column('ID', 'id')
        targets.rename_column('Cluster', 'name')
        targets.rename_column('RA', 'ra')
        targets.rename_column('DEC', 'dec')
        col_formats = [".0f", "", "", "", ".3f"]
    case 3.1: # LAMMIM Cluster Sample (Kluge)
        targets_name = 'lammim_cluster_sample_redMapper_Kluge'
        targets = Table.read("../data/girmos-sci/lammim_cluster_sample_redMapper_Kluge.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.rename_column('z_cl', 'z')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 3.2: # LAMMIM Cluster Sample (Madcowsii)
        targets_name = 'lammim_cluster_sample_madcowsii'
        targets = Table.read("../data/girmos-sci/lammim_cluster_sample_madcowsii.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.rename_column('zspec', 'z')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 3.3: # LAMMIM Cluster Sample (DESI Legacy)
        targets_name = 'lammim_cluster_sample_desi_legacy_wh24'
        targets = Table.read("../data/girmos-sci/lammim_cluster_sample_desi_legacy_wh24.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.rename_column('z_cl', 'z')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 3.4: # LAMMIM lammim_clmem_cat_girmos_cluster_candidates
        targets_name = 'lammim_clmem_cat_girmos_cluster_candidates'
        targets = Table.read("../data/girmos-sci/lammim_clmem_cat_girmos_cluster_candidates.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.remove_column('zwarn')
        targets.remove_column('spectype')
        targets.remove_column('desi_target')
        targets.remove_column('radial_distance')
        targets.remove_column('cl_id')
        targets.remove_column('bgs_flag')
        targets.remove_column('lrg_flag')
        targets.remove_column('elg_flag')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 3.5: # LAMMIM lammim_orelse_clcat_girmos
        targets_name = 'lammim_orelse_clcat_girmos'
        targets = Table.read("../data/girmos-sci/lammim_orelse_clcat_girmos.csv", format='csv')
        targets.remove_column('col0')
        targets.rename_column('Id', 'id')
        targets.remove_column('e_z')
        col_formats = [".0f", ".4f", ".5f", ".5f"]
    case 3.6: # LAMMIM lammim_gal_cat_girmos_candidates
        targets_name = 'lammim_gal_cat_girmos_candidates'
        targets = Table.read("../data/girmos-sci/lammim_gal_cat_girmos_candidates.csv", format='csv')
        targets.rename_column('col0', 'id')
        targets.remove_column('zwarn')
        targets.remove_column('spectype')
        targets.remove_column('desi_target')
        targets.remove_column('bgs_flag')
        targets.remove_column('lrg_flag')
        targets.remove_column('elg_flag')
        col_formats = [".0f", ".5f", ".5f", ".4f"]
    case 4: # Hung et al for Brian
        targets_name = 'hung_et_al_2025'
        targets = Table.read("../data/girmos-sci/Hung_et_al_2025_CDS_Table3.txt", format='ascii.cds')
        targets.rename_column('CID', 'id')
        targets.rename_column('RAdeg', 'ra')
        targets.rename_column('DEdeg', 'dec')
        targets.rename_column('zsys', 'z')
        targets.remove_column('f_CID')
        targets.remove_column('Npeak')
        targets.remove_column('Amp')
        targets.remove_column('e_Amp')
        col_formats = ["", ".0f", ".5f", ".5f", ".4f"]

print('Number of targets:', len(targets))

### Match

In [ ]:
def get_asterism_mags(star1_mag, star2_mag, star3_mag):
    mags = np.full(star1_mag.shape, '', dtype=object)

    mask1 = (star2_mag == -1) & (star3_mag == -1)
    if np.sum(mask1) > 0:
        mags[mask1] = [f"{m1:.1f}" for m1 in star1_mag[mask1]]

    mask2 = (star3_mag == -1) & (~mask1)
    if np.sum(mask2) > 0:
        star_mags = np.stack([star1_mag[mask2], star2_mag[mask2]], axis=1)
        star_mags_sorted = np.sort(star_mags, axis=1)
        mags[mask2] = [f"{m1:.1f},{m2:.1f}" for m1, m2 in zip(star_mags_sorted[:, 0], star_mags_sorted[:, 1])]

    mask3 = ~(mask1 | mask2)
    if np.sum(mask3) > 0:
        star_mags = np.stack([star1_mag[mask3], star2_mag[mask3], star3_mag[mask3]], axis=1)
        star_mags_sorted = np.sort(star_mags, axis=1)
        mags[mask3] = [f"{m1:.1f},{m2:.1f},{m3:.1f}" for m1, m2, m3 in zip(star_mags_sorted[:, 0], star_mags_sorted[:, 1], star_mags_sorted[:, 2])]

    return mags

In [ ]:
asterism_catalog = SkyCoord(ra=asterisms['ra'], dec=asterisms['dec'], unit='deg', frame='icrs')
if isinstance(targets['ra'][0], str) and ':' in targets['ra'][0]:
    targets_catalog = SkyCoord(ra=targets['ra'], dec=targets['dec'], unit=(u.hourangle, u.deg), frame='icrs')
else:
    targets_catalog = SkyCoord(ra=targets['ra'], dec=targets['dec'], unit=(u.deg, u.deg), frame='icrs')

idx, sep, _ = targets_catalog.match_to_catalog_sky(asterism_catalog)

if export_all:
    target_filter = np.ones(len(targets), dtype=bool)
else:
    target_filter = sep < fov/2

distance = sep[target_filter]

matches = targets[target_filter]
matches['asterism_id'] = asterisms['id'][idx][target_filter]
matches['asterism_mags'] = get_asterism_mags(asterisms['star1_mag'][idx][target_filter],asterisms['star2_mag'][idx][target_filter],asterisms['star3_mag'][idx][target_filter])
matches['asterism_ra'] = asterisms['ra'][idx][target_filter]
matches['asterism_dec'] = asterisms['dec'][idx][target_filter]
matches['asterism_distance'] = distance.to(u.arcsec).value
matches['SR_mean'] = asterisms['SR_mean'][idx][target_filter]
matches['SR_min']  = asterisms['SR_min'][idx][target_filter]
matches['SR_max']  = asterisms['SR_max'][idx][target_filter]
matches['EE100_mean'] = asterisms['EE100_mean'][idx][target_filter]
matches['EE100_min']  = asterisms['EE100_min'][idx][target_filter]
matches['EE100_max']  = asterisms['EE100_max'][idx][target_filter]
matches['FWHM_mean'] = asterisms['FWHM_mean'][idx][target_filter]
matches['FWHM_min']  = asterisms['FWHM_min'][idx][target_filter]
matches['FWHM_max']  = asterisms['FWHM_max'][idx][target_filter]

if export_all:
    matches['SR_mean'][distance >= fov/2] = np.nan
    matches['SR_min'][distance >= fov/2] = np.nan
    matches['SR_max'][distance >= fov/2] = np.nan
    matches['EE100_mean'][distance >= fov/2] = np.nan
    matches['EE100_min'][distance >= fov/2] = np.nan
    matches['EE100_max'][distance >= fov/2] = np.nan
    matches['FWHM_mean'][distance >= fov/2] = np.nan
    matches['FWHM_min'][distance >= fov/2] = np.nan
    matches['FWHM_max'][distance >= fov/2] = np.nan

col_formats = col_formats + [".0f", "", ".5f", ".5f", ".1f", ".3f", ".3f", ".3f", ".3f", ".3f", ".3f", ".1f", ".1f", ".1f"]

# Sort with NaNs at the end
if export_all:
    nan_mask = np.isnan(matches['EE100_mean'])

    valid_rows = matches[~nan_mask]
    valid_rows.sort('EE100_mean', reverse=True)

    nan_rows = matches[nan_mask]
    nan_rows.sort('asterism_distance')

    matches = vstack([valid_rows, nan_rows])
else:
    matches.sort('EE100_mean', reverse=True)

print('Number of targets near an asterism:', len(matches))
if len(matches) > 0:
    display(tabulate(matches[:200], headers=matches.colnames, tablefmt='html', floatfmt=col_formats))

### Sky Lines

In [ ]:
max_sky_line_targets = None
if 'z' in matches.colnames and (max_sky_line_targets is None or len(matches) < max_sky_line_targets):
    from astropy.constants import si as constants
    from survey_tools import sky

    R = 3000
    airmass = 1.5 # 1.0 or 1.5 or 2.0
    min_sky_rate = 10 # ph/s/m^2/arcsec^2/nm
    min_sky_trans = 0.8
    sky_trans_multiple = 0.5    # multiple of FWHM
    sky_line_multiple  = 0.5    # multiple of FWHM

    # See: https://classic.sdss.org/dr6/algorithms/linestable.php
    lines = [
        {'name': 'SII'      , 'wavelength_vac': np.array([0.6718290, 0.6732670]), 'sigma': 200},
        {'name': 'Ha'       , 'wavelength_vac': np.array([0.6564610           ]), 'sigma': 200},
        {'name': 'NII'      , 'wavelength_vac': np.array([0.6549890, 0.6585270]), 'sigma': 200},
        {'name': 'Hb'       , 'wavelength_vac': np.array([0.4862680           ]), 'sigma': 200},
        {'name': 'OIII_5008', 'wavelength_vac': np.array([0.4960295, 0.5008240]), 'sigma': 200},
        {'name': 'OIII_4364', 'wavelength_vac': np.array([0.4364436           ]), 'sigma': 200},
        {'name': 'OII'      , 'wavelength_vac': np.array([0.3727092, 0.3729875]), 'sigma': 200},
        {'name': 'MgI b1'   , 'wavelength_vac': np.array([0.5183604           ]), 'sigma': 200},
        {'name': 'CaII H'   , 'wavelength_vac': np.array([0.3969588           ]), 'sigma': 200},
        {'name': 'CaII K'   , 'wavelength_vac': np.array([0.3934777           ]), 'sigma': 200},
        {'name': 'NaI D'    , 'wavelength_vac': np.array([0.5891583, 0.5897558]), 'sigma': 200},
    ]

    for l in lines:
        l['wavelength'] = sky.get_vacuum_to_air_wavelength(l['wavelength_vac']*u.micron).value
        matches[l['name']] = False
        col_formats.append(".0f")

    match R:
        case 3000:
            bands = [
                {'name': 'YJ', 'start': 0.95, 'end': 1.35},
                {'name': 'JH', 'start': 1.25, 'end': 1.80},
                {'name': 'HK', 'start': 1.63, 'end': 2.35},
            ]
        case 8000:
            bands = [
                {'name': 'Js', 'start': 1.194, 'end': 1.350},
                {'name': 'Hs', 'start': 1.500, 'end': 1.706},
                {'name': 'Ks', 'start': 2.110, 'end': 2.379},
            ]

    band_wavelength_range  = np.array([[b['start'], b['end']] for b in bands])
    band_wavelength_min = np.min(band_wavelength_range)
    band_wavelength_max = np.max(band_wavelength_range)
    print(f"Bands: {band_wavelength_min}-{band_wavelength_max} μm (R={R})")

    sky_transmission_data = sky.load_transmission_data('MaunaKea', airmass)
    print(f"Sky Transmission: {sky_transmission_data['wavelength'][0]/10:.0f}-{sky_transmission_data['wavelength'][-1]/10:.0f} nm, N={len(sky_transmission_data)}, dλ = {(sky_transmission_data['wavelength'][1]-sky_transmission_data['wavelength'][0])/10:.2f} nm")

    sky_background_data = sky.load_background_data('MaunaKea', airmass)
    print(f"Sky Background: {sky_background_data['wavelength'][0]/10:.0f}-{sky_background_data['wavelength'][-1]/10:.0f} nm, N={len(sky_background_data)}, dλ = {(sky_background_data['wavelength'][1]-sky_background_data['wavelength'][0])/10:.2f} nm")

    last_time = time.time()
    print("Rejecting emission lines...")
    for i, match in enumerate(matches):
        z = match['z']

        for l in lines:
            w = l['wavelength'] * (1 + z)
            fwhm = np.sqrt((w/R)**2 + (w * 2.35482 * l['sigma'] / constants.c.to('km/s').value)**2)

            reject = sky.reject_emission_line(
                sky_background_data,
                sky_transmission_data,
                w*1e4, # convert micron to angstrom
                fwhm*1e4, # convert micron to angstrom
                R,
                allowed_wavelength_range=band_wavelength_range*1e4, # convert micron to angstrom
                trans_minimum=min_sky_trans, 
                trans_dLambda_multiple=sky_trans_multiple, 
                avoid_dLambda_multiple=sky_line_multiple, 
                min_photon_rate=min_sky_rate
            )

            match[l['name']] = not np.all(reject)

        if len(matches) > 1000:
            if (i+1) % 1000 == 0:
                elapsed_time = time.time() - last_time
                last_time = time.time()
                print(f"  {i+1}/{len(matches)} ({elapsed_time:.2f}s)")

### Export Results

In [ ]:
matches.write(f"../output/matches-{targets_name}.csv", format='csv', overwrite=True)